# Building a web crawler
* lesson 25 already scraped a SINGLE page with `requests` + `BeautifulSoup` - a **crawler** goes one step further: it automatically follows the links it finds and scrapes MANY pages, one after another
* this is exactly how search engines (Google, Bing) originally build their index, and its also how price-comparison sites, archive tools and research scrapers work
* just like lesson 25, we build our OWN tiny local multi-page "website" to crawl - this keeps the lesson 100% reliable offline, and its simply more polite than pointing a crawling exercise at somebody else's real server

## building a tiny local multi-page website to crawl
* our mini site has a homepage linking to 3 sub-pages, and those sub-pages link to each other and back to the homepage (a small, realistic link graph, including a cycle - a real crawler MUST be able to handle that without looping forever)
* we also serve a `robots.txt` file, exactly like a real website would - more on that in a moment
* same reliable pattern as lesson 14/25: `http.server` + `threading`, bind port 0, daemon thread, silence per-request logging, clean shutdown at the end

In [1]:
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer

# our tiny "website" - a dict mapping a path to its raw html content
# notice how "/" links to /about and /blog, /blog links to /post1 and /post2, and /post1 links BACK to "/" - a cycle!
PAGES = {
    "/": """
        <html><head><title>Home</title></head>
        <body>
            <h1>Welcome</h1>
            <p>This is the homepage.</p>
            <a href="/about">About us</a>
            <a href="/blog">Blog</a>
        </body></html>
    """,
    "/about": """
        <html><head><title>About</title></head>
        <body>
            <h1>About</h1>
            <p>We are a small fictional company, built purely for this lesson.</p>
            <a href="/">Back home</a>
        </body></html>
    """,
    "/blog": """
        <html><head><title>Blog</title></head>
        <body>
            <h1>Blog</h1>
            <a href="/blog/post1">First post</a>
            <a href="/blog/post2">Second post</a>
            <a href="/">Home</a>
        </body></html>
    """,
    "/blog/post1": """
        <html><head><title>Post 1</title></head>
        <body>
            <h1>First post</h1>
            <p>Some example blog content here.</p>
            <a href="/">Home</a>
            <a href="/blog/post2">Next post</a>
        </body></html>
    """,
    "/blog/post2": """
        <html><head><title>Post 2</title></head>
        <body>
            <h1>Second post</h1>
            <p>More example blog content.</p>
            <a href="/blog/post1">Previous post</a>
            <a href="/blog/secret">A page we dont want crawled</a>
        </body></html>
    """,
    # a page that exists, but robots.txt (below) will explicitly disallow crawling it - a well-behaved crawler must skip it
    "/blog/secret": """
        <html><head><title>Secret</title></head>
        <body><h1>Shhh</h1><p>This page is disallowed by robots.txt.</p></body></html>
    """,
}

# a robots.txt file, exactly like a real website would serve - it tells (well-behaved) crawlers what they may/may not visit
ROBOTS_TXT = """
User-agent: *
Disallow: /blog/secret
Crawl-delay: 0
"""


class MiniSiteHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/robots.txt":
            body = ROBOTS_TXT.encode("utf-8")
            content_type = "text/plain"
        elif self.path in PAGES:
            body = PAGES[self.path].encode("utf-8")
            content_type = "text/html"
        else:
            self.send_response(404)
            self.end_headers()
            self.wfile.write(b"not found")
            return

        self.send_response(200)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # keep the notebook output clean, same as earlier lessons


site_server = HTTPServer(("127.0.0.1", 0), MiniSiteHandler)
site_port = site_server.server_address[1]
site_thread = threading.Thread(target=site_server.serve_forever, daemon=True)
site_thread.start()

BASE_URL = f"http://127.0.0.1:{site_port}"
print(f"tiny crawlable website running on {BASE_URL}")

tiny crawlable website running on http://127.0.0.1:42725


## robots.txt - the polite crawler's first stop
* before scraping ANY page on a real site, a well-behaved crawler checks `https://thesite.com/robots.txt` first
* it's a plain text file that tells crawlers which paths they may (`Allow`) or may not (`Disallow`) visit, and sometimes a `Crawl-delay` (minimum seconds to wait between requests to that site)
* robots.txt is a VOLUNTARY convention, not a technical lock - nothing stops code from ignoring it, but ignoring it is considered rude at best and can get your crawler's IP banned (or worse, be a real legal problem) at worst
* python's standard library ships `urllib.robotparser` specifically to parse and check robots.txt for us, no extra install needed

In [2]:
from urllib.robotparser import RobotFileParser

robots_parser = RobotFileParser()
robots_parser.set_url(f"{BASE_URL}/robots.txt")  # tell it WHERE to fetch robots.txt from
robots_parser.read()  # this actually downloads and parses it (uses urllib internally)

# can_fetch(user_agent, url) is the one method we actually need - it answers "am I allowed to crawl this?"
print("can we crawl /about?      ", robots_parser.can_fetch("*", f"{BASE_URL}/about"))
print("can we crawl /blog/post1? ", robots_parser.can_fetch("*", f"{BASE_URL}/blog/post1"))
print("can we crawl /blog/secret?", robots_parser.can_fetch("*", f"{BASE_URL}/blog/secret"))  # False! disallowed above

# our crawler below will call can_fetch() on every single url BEFORE fetching it, and simply skip anything disallowed

can we crawl /about?       True
can we crawl /blog/post1?  True
can we crawl /blog/secret? False


## the crawler itself: a breadth-first traversal of the link graph
* a crawler is really just a graph traversal (if you did the "which sorting/searching algorithm" flavor of computer science courses, this is breadth-first search) over the "graph" where pages are nodes and links are edges
* we need: a `frontier` (a queue of urls still to visit, we use `collections.deque` from lesson 12 for an efficient FIFO queue), and a `visited` set (so we never re-crawl the same url twice, which would otherwise loop forever thanks to that "post1 links back home" cycle we built on purpose)
* for every page we visit: fetch it, parse out every link with `BeautifulSoup`, turn each into a full, absolute url with `urllib.parse.urljoin` (a link might be written as just `/about`, relative to the CURRENT page), keep only same-site links, and if we havent seen it yet AND robots.txt allows it, add it to the frontier

In [3]:
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from collections import deque

def crawl(start_url, max_pages=20, delay=0.05):
    """Breadth-first crawl starting at start_url, staying on the same host, respecting robots.txt."""
    frontier = deque([start_url])   # deque.popleft() is O(1), perfect for a FIFO "visit in the order we discovered them" queue
    visited = set()                  # never process the same url twice
    pages_crawled = []               # our collected results: one dict per successfully crawled page
    start_host = urlparse(start_url).netloc  # e.g. "127.0.0.1:44444" - we only want to follow links staying on THIS host

    while frontier and len(pages_crawled) < max_pages:
        url = frontier.popleft()  # take the OLDEST discovered url off the queue (breadth-first, not depth-first)

        if url in visited:
            continue
        visited.add(url)  # mark it visited BEFORE fetching, so even a slow/failing fetch never gets retried in this run

        if not robots_parser.can_fetch("*", url):
            print(f"skipping (robots.txt disallows): {url}")
            continue

        response = requests.get(url, timeout=5)
        if response.status_code != 200:
            print(f"skipping (status {response.status_code}): {url}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        title = soup.title.get_text() if soup.title else "(no title)"
        pages_crawled.append({"url": url, "title": title})
        print(f"crawled: {url}  ->  \"{title}\"")

        # find every link on this page and, if it qualifies, queue it up for a future visit
        for link_tag in soup.find_all("a", href=True):
            absolute_link = urljoin(url, link_tag["href"])  # turn a relative link like "/about" into a full url
            if urlparse(absolute_link).netloc == start_host and absolute_link not in visited:
                frontier.append(absolute_link)  # add to the END of the queue - this is what makes it breadth-first

        time.sleep(delay)  # politeness: dont hammer the server with requests as fast as our code can possibly go

    return pages_crawled


results = crawl(BASE_URL, max_pages=20)
print()
print(f"crawled {len(results)} pages in total (and correctly skipped /blog/secret because of robots.txt)")

crawled: http://127.0.0.1:42725  ->  "Home"
crawled: http://127.0.0.1:42725/about  ->  "About"


crawled: http://127.0.0.1:42725/blog  ->  "Blog"


crawled: http://127.0.0.1:42725/  ->  "Home"


crawled: http://127.0.0.1:42725/blog/post1  ->  "Post 1"
crawled: http://127.0.0.1:42725/blog/post2  ->  "Post 2"


skipping (robots.txt disallows): http://127.0.0.1:42725/blog/secret

crawled 6 pages in total (and correctly skipped /blog/secret because of robots.txt)


## storing the results properly
* a real crawler usually collects way more than just the title - but the pattern (build a list of dicts, one per page) is exactly the same no matter how many fields you extract
* once we have that list of dicts, dropping it straight into a pandas DataFrame (recap lesson 19) gives us a nice, sortable/filterable table of everything we found

In [4]:
import pandas as pd

results_df = pd.DataFrame(results)
print(results_df)

# now we can use everything we learned in lesson 19, for example: which page titles contain "post"?
print()
print("pages with 'post' in the title:")
print(results_df[results_df["title"].str.contains("post", case=False)])

                                 url   title
0             http://127.0.0.1:42725    Home
1       http://127.0.0.1:42725/about   About
2        http://127.0.0.1:42725/blog    Blog
3            http://127.0.0.1:42725/    Home
4  http://127.0.0.1:42725/blog/post1  Post 1
5  http://127.0.0.1:42725/blog/post2  Post 2

pages with 'post' in the title:
                                 url   title
4  http://127.0.0.1:42725/blog/post1  Post 1
5  http://127.0.0.1:42725/blog/post2  Post 2


## going faster: crawling concurrently
* our crawler above fetches one page, waits, fetches the next, waits, ... completely sequential
* since fetching a page is almost entirely WAITING on the network (I/O-bound, recap lesson 11), this is a perfect case for `concurrent.futures.ThreadPoolExecutor` - we can have several requests "in flight" to different pages at once
* important nuance: the *frontier discovery* (finding new links) still has to happen level by level (we dont know a page's links until we've fetched it) - but ONCE we already know a batch of urls (like right at the start, or all children of one page), fetching THAT batch concurrently is a straightforward, safe win

In [5]:
from concurrent.futures import ThreadPoolExecutor

# lets fetch every page of our tiny site concurrently, now that we already know all their urls from the crawl above
all_known_urls = [f"{BASE_URL}{path}" for path in PAGES if robots_parser.can_fetch("*", f"{BASE_URL}{path}")]
print("fetching", len(all_known_urls), "pages concurrently:", all_known_urls)

def fetch_title(url):
    response = requests.get(url, timeout=5)
    soup = BeautifulSoup(response.text, "html.parser")
    return url, (soup.title.get_text() if soup.title else "(no title)")

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as executor:
    concurrent_results = list(executor.map(fetch_title, all_known_urls))
elapsed = time.perf_counter() - start

for url, title in concurrent_results:
    print(f"  {url} -> \"{title}\"")
print(f"fetched all {len(concurrent_results)} pages concurrently in {elapsed:.3f}s")

# on a real, larger crawl (thousands of pages, real network latency) this kind of concurrency is the difference between
# a crawl that takes hours and one that takes minutes - real crawling frameworks like Scrapy build this in by default

fetching 5 pages concurrently: ['http://127.0.0.1:42725/', 'http://127.0.0.1:42725/about', 'http://127.0.0.1:42725/blog', 'http://127.0.0.1:42725/blog/post1', 'http://127.0.0.1:42725/blog/post2']
  http://127.0.0.1:42725/ -> "Home"
  http://127.0.0.1:42725/about -> "About"
  http://127.0.0.1:42725/blog -> "Blog"
  http://127.0.0.1:42725/blog/post1 -> "Post 1"
  http://127.0.0.1:42725/blog/post2 -> "Post 2"
fetched all 5 pages concurrently in 0.019s


## a note on real-world crawling tools
* for anything beyond a small learning exercise, most python developers reach for **Scrapy** - a full, dedicated crawling framework (handles the frontier/queue, concurrency, retries, robots.txt, rate-limiting, exporting to csv/json, and much more, all built in) rather than hand-rolling the loop we just wrote - we didn't install it here to keep this lesson dependency-light, but everything we just built by hand is exactly the set of concepts Scrapy automates for you under the hood
* for crawling JavaScript-heavy modern websites (where the real content only appears after JS runs in a browser, `requests` alone cant see it), python developers reach for a **headless browser** automation tool like `playwright` or `selenium` instead - these actually launch a real (invisible) browser engine to render the page first
* we keep our little local mini-site running for a while longer - the exercises below still need it, we shut it down properly at the very end of this lesson

## an important legal/ethical note
* always check a real site's `robots.txt` AND its terms of service before crawling it for real - some sites explicitly forbid scraping/crawling in their ToS even if robots.txt is permissive
* respect `Crawl-delay` and generally rate-limit yourself even if a site doesn't specify one - hammering someone else's server with requests can look like (or effectively become) a denial-of-service attack, and can get your IP address blocked
* set a real, honest `User-Agent` header identifying your crawler/bot (many sites expect this and may block generic/anonymous-looking traffic) and, ideally, a way for the site owner to contact you if there's a problem
* never crawl content behind a login wall or paywall unless you have explicit permission - just because your browser session can see it doesn't mean an automated script is allowed to

## quick comparison to other languages
* Scrapy (python) is the most popular dedicated crawling framework, but nearly every ecosystem has an equivalent: **Apache Nutch** (Java, one of the original open-source web-scale crawlers), **Colly** (Go, popular for its speed and simplicity), Node.js has both scraping libraries (`cheerio`, very similar in spirit to BeautifulSoup) and full crawling frameworks (`crawlee`)
* for JS-heavy sites needing a real rendered browser, **Playwright** and **Puppeteer** both offer near-identical APIs in python, JavaScript/TypeScript, and (Playwright only) Java/C# - browser automation is genuinely one of the most cross-language-consistent tooling spaces out there
* the underlying concepts (frontier/queue, visited set, respecting robots.txt, rate limiting, concurrency for speed) are 100% language-agnostic - this whole lesson would look almost identical structurally if rewritten in Go or JavaScript, only the http-client/HTML-parsing library names would change

## Exercises
1. modify the `crawl()` function to also collect, per page, how many links it found on that page (not just the title)
2. add a `max_depth` parameter to `crawl()` that stops following links beyond a certain "distance" from the start page (hint: track depth alongside each url in the frontier, e.g. store `(url, depth)` tuples instead of plain urls)
3. write a small check that counts how many pages listed in the `PAGES` dict were correctly SKIPPED because of `robots.txt`, using `robots_parser.can_fetch()`
4. use `ThreadPoolExecutor` to concurrently check `can_fetch()`-allowed status for every url in `PAGES` and print a dict of `{url: allowed_or_not}`

In [6]:
# TODO: solve exercise 1 here (also count links per page)


# TODO: solve exercise 2 here (add a max_depth parameter to crawl())


# TODO: solve exercise 3 here (count pages skipped by robots.txt)


# TODO: solve exercise 4 here (concurrently check can_fetch() for every url in PAGES)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [7]:
# sample solution 1 - also count links per page
def crawl_with_link_count(start_url, max_pages=20, delay=0.02):
    frontier = deque([start_url])
    visited = set()
    pages_crawled = []
    start_host = urlparse(start_url).netloc

    while frontier and len(pages_crawled) < max_pages:
        url = frontier.popleft()
        if url in visited:
            continue
        visited.add(url)
        if not robots_parser.can_fetch("*", url):
            continue
        response = requests.get(url, timeout=5)
        if response.status_code != 200:
            continue
        soup = BeautifulSoup(response.text, "html.parser")
        title = soup.title.get_text() if soup.title else "(no title)"
        links_found = soup.find_all("a", href=True)
        pages_crawled.append({"url": url, "title": title, "link_count": len(links_found)})
        for link_tag in links_found:
            absolute_link = urljoin(url, link_tag["href"])
            if urlparse(absolute_link).netloc == start_host and absolute_link not in visited:
                frontier.append(absolute_link)
        time.sleep(delay)
    return pages_crawled

print(pd.DataFrame(crawl_with_link_count(BASE_URL)))
print()

# sample solution 2 - add a max_depth parameter
def crawl_with_depth_limit(start_url, max_depth=1, delay=0.02):
    frontier = deque([(start_url, 0)])  # store (url, depth) pairs instead of just urls
    visited = set()
    pages_crawled = []
    start_host = urlparse(start_url).netloc

    while frontier:
        url, depth = frontier.popleft()
        if url in visited or depth > max_depth:
            continue
        visited.add(url)
        if not robots_parser.can_fetch("*", url):
            continue
        response = requests.get(url, timeout=5)
        if response.status_code != 200:
            continue
        soup = BeautifulSoup(response.text, "html.parser")
        pages_crawled.append({"url": url, "depth": depth})
        if depth < max_depth:  # only keep exploring further if we havent hit the depth limit yet
            for link_tag in soup.find_all("a", href=True):
                absolute_link = urljoin(url, link_tag["href"])
                if urlparse(absolute_link).netloc == start_host and absolute_link not in visited:
                    frontier.append((absolute_link, depth + 1))
        time.sleep(delay)
    return pages_crawled

print(pd.DataFrame(crawl_with_depth_limit(BASE_URL, max_depth=1)))  # only the homepage + its DIRECT links
print()

# sample solution 3 - count pages skipped by robots.txt
skipped_count = sum(1 for path in PAGES if not robots_parser.can_fetch("*", f"{BASE_URL}{path}"))
print(f"{skipped_count} out of {len(PAGES)} pages are disallowed by robots.txt")
print()

# sample solution 4 - concurrently check can_fetch() for every url
all_paths = list(PAGES.keys())
with ThreadPoolExecutor(max_workers=5) as executor:
    # can_fetch() itself doesnt need to be threaded (its just checking already-downloaded rules, no new network call)
    # but this shows the *pattern* of mapping a function concurrently over many inputs, which DOES matter once the check involves real IO
    allowed_map = dict(executor.map(lambda path: (path, robots_parser.can_fetch("*", f"{BASE_URL}{path}")), all_paths))
print(allowed_map)

                                 url   title  link_count
0             http://127.0.0.1:42725    Home           2
1       http://127.0.0.1:42725/about   About           1
2        http://127.0.0.1:42725/blog    Blog           3
3            http://127.0.0.1:42725/    Home           2
4  http://127.0.0.1:42725/blog/post1  Post 1           2
5  http://127.0.0.1:42725/blog/post2  Post 2           2



                            url  depth
0        http://127.0.0.1:42725      0
1  http://127.0.0.1:42725/about      1
2   http://127.0.0.1:42725/blog      1

1 out of 6 pages are disallowed by robots.txt

{'/': True, '/about': True, '/blog': True, '/blog/post1': True, '/blog/post2': True, '/blog/secret': False}


## cleanup
* now that every section (including the exercises) is done, its time to shut down our little local website, exactly like the previous lessons taught us

In [8]:
site_server.shutdown()
site_thread.join(timeout=2)
print("crawlable mini-site shut down, thread alive?", site_thread.is_alive())

crawlable mini-site shut down, thread alive? False


* congratulation you finished this lesson

## what we learned
* the difference between scraping ONE page (lesson 25) and crawling MANY pages by following links automatically
* `robots.txt` and `urllib.robotparser` - checking what a site allows a crawler to visit before fetching it
* a breadth-first crawler: a `deque` frontier queue + a `visited` set, turning relative links into absolute urls with `urljoin`, and staying on the same host
* storing crawl results in a pandas DataFrame for easy filtering/analysis
* speeding up crawling with `ThreadPoolExecutor` since fetching pages is I/O-bound work
* real-world crawling tools (Scrapy for large crawls, Playwright/Selenium for JavaScript-heavy sites) and why they exist
* the legal/ethical responsibilities of running a crawler against a real website